<a href="https://colab.research.google.com/github/ripims1/COMP-3608-PROJECT/blob/tyler-data-blending/Model_Ensemble.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
# Global library declaration
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt


In [ ]:
# ========================
#  Load and Prepping Data
# ========================

#Load the data
train_df = pd.read_csv('train_processed.csv')
test_df = pd.read_csv('test_processed.csv')

#Separate Features (X) and Target (y)
X_train_raw = train_df.drop('Churn', axis=1).values
y_train_raw = train_df['Churn'].values
X_test_raw = test_df.values

# Splitting training data to use for training and evaluation
X_train_main, X_val, y_train_main, y_val = train_test_split(
    X_train_raw, y_train_raw, test_size=0.2, random_state=42
)


In [ ]:
# ==============
# Neural Network
# ==============

#Scaling
scaler = StandardScaler()

# Fit scalar only on training data
X_train_main_scaled = scaler.fit_transform(X_train_main)

# Same transformation applied to validation and test sets
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test_raw)

#Convert to Tensors
X_train_tensor = torch.tensor(X_train_main_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_main, dtype=torch.float32).reshape(-1, 1)
X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

print(f"X_train shape: {X_train_tensor.shape}")
print(f"y_train shape: {y_train_tensor.shape}")

class ChurnModel(nn.Module):
    def __init__(self, input_size):
        super(ChurnModel, self).__init__()

        #Layer 1: Input (45 features) -> 64 Hidden Neurons
        self.fc1 = nn.Linear(input_size, 64)

        #Layer 2: 64 Neurons -> 32 Hidden Neurons
        self.fc2 = nn.Linear(64, 32)

        #Layer 3: 32 Neurons -> 1 Output
        self.fc3 = nn.Linear(32, 1)

    def forward(self, x):
        #Pass through Layer 1 and apply ReLU activation
        x = F.relu(self.fc1(x))

        #Pass through Layer 2 and apply ReLU activation
        x = F.relu(self.fc2(x))

        #Output Layer: Use Sigmoid to get a value between 0 and 1
        x = torch.sigmoid(self.fc3(x))

        return x

#Initialize the model using the number of columns (45)
input_dim = X_train_tensor.shape[1]
model = ChurnModel(input_dim)

print(model)

#Define the Loss Function
criterion = nn.BCELoss()

#Define the Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Loss function and Optimizer are ready.")

#Combine X and y into a single Dataset object
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)

#Create the DataLoader
train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)

print("DataLoader is ready.")

epochs = 10

for epoch in range(epochs):
    model.train() #Set model to training mode
    running_loss = 0.0

    for inputs, labels in train_loader:
        #Clear the old gradients
        optimizer.zero_grad()

        #Forward Pass: Get predictions
        outputs = model(inputs)

        #Calculate Loss
        loss = criterion(outputs, labels)

        #Backward Pass
        loss.backward()

        #Optimizer Step: Update the weights
        optimizer.step()

        running_loss += loss.item()

    #Print progress after each epoch
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")

print("Training Complete!")

#Switch to evaluation mode
model.eval()


with torch.no_grad():

    # Get predictions on validation set
    nn_val_outputs = model(X_val_tensor)

    # Convert to numpy and flatten for AUC calculation
    nn_val_preds = nn_val_outputs.numpy().flatten()

with torch.no_grad():

    # Get predictions on test set
    test_outputs = model(X_test_tensor)

    # Convert to numpy array and flatten
    nn_test_preds = test_outputs.numpy().flatten()

# Value for blending
prediction_NN = nn_test_preds

In [13]:
# ==========
#  LightGBM
# ==========

# Use same scaled datasets as in the NN model
X_train_lgb = X_train_main
Y_train_lgb = y_train_main
X_val_lgb = X_val
X_test_lgb = X_test_raw

# Convert the training and validation data to the LightGBM datset format
train_data = lgb.Dataset(X_train_lgb, label=Y_train_lgb)
val_data = lgb.Dataset(X_val_lgb, label=y_val)

# Define the model parametrs for LightGBM to control how it learns
params = {
    'objective': 'binary',        # Binary Classification problem
    'metric': 'auc',              # Area under ROC curve
    'boosting_type': 'gbdt',      # Gradient Boosting decision trees
    'learning_rate': 0.03,        # Learning step size
    'num_leaves': 64,             # Controls complexity of trees
    'feature_fraction': 0.8,      # Randomly select features to prevent overfitting
    'bagging_fraction': 0.8,      # Randomly sample data
    'bagging_freq': 5,            # Frequency of bagging
    'verbosity': -1               # Suppress training output
}

# Train the model using the dataset
lgb_model = lgb.train(params,                               # All model parameters
                      train_data,                           # Training Dataset
                      num_boost_round=1000,                 # Number of trees to build
                      valid_sets=[val_data],                # Validation dataset to evalulate during training
                      callbacks=[lgb.early_stopping(50)]    # Stop training if validation auc doesnt improve for 50 rounds
                      )

# Generate probability predictions for the validation dataset
lgb_val_preds = lgb_model.predict(X_val_lgb)

# Generate probability predictions for the test dataset and output between 0 and 1
prediction_LGBM = lgb_model.predict(X_test_lgb)

print("Successfully generated prediction")

In [17]:
# ==============
# Model Blending
# ==============

# Blends predictions in a 85 15
final_prediction = 0.85 * prediction_LGBM + 0.15 * prediction_NN

In [18]:
# ==================
#  FINAL SUBMISSION
# ==================

start_id = 594194
end_id = 848848

submission_df = pd.DataFrame({
    'id': range(start_id, end_id + 1),
    'Churn': final_prediction
})

submission_df.to_csv('submission5.csv', index=False)

print("Blended submission file created!")

Blended submission file created!
